Filter recipe types

In [38]:
# Paths to your files
file_path = './khairulamingDOTmyURLs.txt'
output_path = './filteredURL.txt'
keyword = "daging"  # Replace with your desired keyword

# Open the source file to read, and the output file to write
with open(file_path, 'r', encoding='utf-8') as infile, open(output_path, 'w', encoding='utf-8') as outfile:
    for line in infile:
        url = line.strip()
        
        # Check for your keyword AND make sure "#content" is NOT in the URL
        if keyword in url.lower() and "#content" not in url.lower():
            # Write the URL to the new file, adding a newline character at the end
            outfile.write(url + '\n')

print(f"Done! Filtered URLs have been saved to '{output_path}'.")

Done! Filtered URLs have been saved to './filteredURL.txt'.


Scrape em - filtered_descriptions.json

In [39]:
import json
import re
import time
from bs4 import BeautifulSoup
import requests

input_path = "./filteredURL.txt"
output_json_path = "./filtered_descriptions.json"

output_data = {"recipes": []}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

with open(input_path, "r", encoding="utf-8") as file:
    urls = [line.strip() for line in file if line.strip()]

total_urls = len(urls)
print(f"🚀 Starting fixed verbose scrape of {total_urls} recipes...\n" + "=" * 50)

for index, url in enumerate(urls, 1):
    slug = url.strip("/").split("/")[-1]
    recipe_id = slug
    recipe_title = slug.replace("-", " ").title()

    print(f"\n[#{index}/{total_urls}] Processing: {recipe_title}")
    print(f" 🔗 URL: {url}")

    try:
        response = requests.get(url, headers=headers, timeout=10)
        ingredients_list = []
        video_url = url  # Fallback

        if response.status_code == 200:
            print(" 📂 Page source downloaded successfully (HTTP 200).")
            soup = BeautifulSoup(response.text, "html.parser")

            # --- Extract the Instagram Link from tags/hrefs ---
            # Search all elements for text/links matching an instagram post/reel
            ig_match = None
            
            # Strategy A: Check blockquotes or anchors containing an Instagram URL
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if "instagram.com" in href and ("/p/" in href or "/reel/" in href):
                    ig_match = href
                    break
            
            # Strategy B: If no link found, look through the text/raw data attributes
            if not ig_match:
                blockquote = soup.find(class_="instagram-media")
                if blockquote and blockquote.get("data-instgrm-permalink"):
                    ig_match = blockquote["data-instgrm-permalink"]

            if ig_match:
                # Extract clean link up to the shortcode (removes query strings)
                match = re.match(r"(https://www\.instagram\.com/(?:p|reel)/[^/?]+)", ig_match)
                if match:
                    video_url = match.group(1) + "/"
                    print(f" 🎥 Found Instagram Link: {video_url}")
                else:
                    video_url = ig_match
                    print(f" 🎥 Found Instagram Link (Raw): {video_url}")
            else:
                print(" ℹ️ No Instagram embed details detected in static HTML. Using page URL fallback.")

            # --- Extract Ingredients ---
            all_text_lines = []
            for p in soup.find_all("p"):
                for string in p.stripped_strings:
                    all_text_lines.append(string)

            found_start = False
            for line in all_text_lines:
                # Broadened check: Matches "bahan", "bahan-bahan", "bahan daging", etc.
                if "bahan" in line.lower():
                    found_start = True
                    # If the marker contains the actual ingredients on the same line, don't skip it
                    if len(line) > 20: 
                        ingredients_list.append(line)
                    continue

                if found_start:
                    # Stop markers
                    if (
                        "resipi lain" in line.lower()
                        or "sebelumnya" in line.lower()
                        or "seterusnya" in line.lower()
                    ):
                        break
                    ingredients_list.append(line)

            if ingredients_list:
                print(f" 📋 Successfully extracted {len(ingredients_list)} rows of ingredients.")
            else:
                print(" ❌ Failed to parse any items under 'bahan' markers.")

        else:
            print(f" ❌ Failed to load page. HTTP Status Code: {response.status_code}")

        # Build the final object
        recipe_entry = {
            "id": recipe_id,
            "title": recipe_title,
            "source": "khairul-aming",
            "videoUrl": video_url,
            "ingredients": ingredients_list if ingredients_list else ["Could not parse ingredients automatically."],
            "prepTime": "45 min",
            "servings": 4,
        }
        output_data["recipes"].append(recipe_entry)

    except Exception as e:
        print(f" 🔥 Critical Error processing {url}: {e}")
        output_data["recipes"].append(
            {
                "id": recipe_id,
                "title": recipe_title,
                "source": "khairul-aming",
                "videoUrl": url,
                "ingredients": [f"Error fetching page: {str(e)}"],
                "prepTime": "N/A",
                "servings": 0,
            }
        )

    time.sleep(1)

print("\n" + "=" * 50)
with open(output_json_path, "w", encoding="utf-8") as json_file:
    json.dump(output_data, json_file, ensure_ascii=False, indent=4)

print(f"🎉 Scraping complete! Structured payload written to '{output_json_path}'.")

🚀 Starting fixed verbose scrape of 10 recipes...

[#1/10] Processing: Daging Dendeng
 🔗 URL: https://khairulaming.my/daging-dendeng/
 📂 Page source downloaded successfully (HTTP 200).
 🎥 Found Instagram Link: https://www.instagram.com/reel/CrC60MkgBdE/
 📋 Successfully extracted 13 rows of ingredients.

[#2/10] Processing: Rendang 2 Kilo Daging
 🔗 URL: https://khairulaming.my/rendang-2-kilo-daging/
 📂 Page source downloaded successfully (HTTP 200).
 🎥 Found Instagram Link: https://www.instagram.com/reel/C5U-BO4PiTH/
 📋 Successfully extracted 18 rows of ingredients.

[#3/10] Processing: Kari Daging Raya Haji
 🔗 URL: https://khairulaming.my/kari-daging-raya-haji/
 📂 Page source downloaded successfully (HTTP 200).
 🎥 Found Instagram Link: https://www.instagram.com/reel/Cfsm6rZgXt1/
 📋 Successfully extracted 12 rows of ingredients.

[#4/10] Processing: Daging Balado
 🔗 URL: https://khairulaming.my/daging-balado/
 📂 Page source downloaded successfully (HTTP 200).
 🎥 Found Instagram Link: htt

filtered recipe


In [41]:
import json

# Define all file paths
ingredients_master_path = r"C:\GetSTACKED2024\masakapa\src\data\ingredients.json"
scraped_json_path = "./filtered_descriptions.json"
output_recipe_path = "./filtered_recipe.json"

# 1. Load your master ingredients database
print("📋 Loading master ingredients database...")
with open(ingredients_master_path, "r", encoding="utf-8") as file:
    master_data = json.load(file)

# Build lookup dictionary: {"lowercase name": "id"}
ingredient_lookup = {}
for ing in master_data["ingredients"]:
    name_lower = ing["name"].lower().strip()
    ingredient_lookup[name_lower] = ing["id"]

# 2. Load the scraped recipe description file
with open(scraped_json_path, "r", encoding="utf-8") as file:
    scraped_data = json.load(file)

final_data = {"recipes": []}
total_recipes = len(scraped_data["recipes"])

print(f"🔄 Starting verbose mapping for {total_recipes} recipes...\n" + "="*60)

# 3. Match scraped ingredients against master IDs
for index, recipe in enumerate(scraped_data["recipes"], 1):
    print(f"\n[#{index}/{total_recipes}] Mapping: {recipe['title']}")
    matched_ingredients = []

    # Skip right away if it's an error marker block before we waste time processing lines
    if not recipe["ingredients"] or "error" in recipe["ingredients"][0].lower() or "could not parse" in recipe["ingredients"][0].lower():
        print("  ❌ Skipping recipe: Contains error or empty markers.")
        continue

    for ing_text in recipe["ingredients"]:
        line = ing_text.strip()
        line_lower = line.lower()

        # Skip layout headers
        if "bahan" in line_lower or "cara" in line_lower or "untuk" in line_lower:
            continue

        matched_id = None

        # --- Hardcoded Tweaks & Substitution Logic ---
        if any(rempah in line_lower for rempah in ["pelaga", "bunga lawang", "cengkih", "kayu manis"]):
            matched_id = "booby-trap"
        elif "bawang holland" in line_lower:
            # Fallback to 'bawang-besar' if 'bawang' doesn't exist explicitly in your master JSON ids
            matched_id = ingredient_lookup.get("bawang besar", "bawang")
        elif "serbuk perasa" in line_lower:
            matched_id = "ajinomoto"

        # --- Standard Master Dictionary Lookup ---
        if not matched_id:
            # Sort by length descending to match long phrases first ('kicap manis' before 'kicap')
            for master_name in sorted(ingredient_lookup.keys(), key=len, reverse=True):
                if master_name in line_lower:
                    matched_id = ingredient_lookup[master_name]
                    break

        # --- General Fallback Regex Rules ---
        if not matched_id:
            if "cili padi" in line_lower or "cili api" in line_lower:
                matched_id = "cili-padi"
            elif "daging" in line_lower:
                matched_id = "daging"

        # Resolve entity match status
        if matched_id:
            print(f"  ✅ [MATCH] '{line}' ➡️ '{matched_id}'")
            if matched_id not in matched_ingredients:
                matched_ingredients.append(matched_id)
        else:
            # Wrap no-match items with an asterisk
            asterisk_fallback = f"*{line}*"
            print(f"  ⚠️  [NO MATCH] '{line}' ➡️ Keeping raw as: {asterisk_fallback}")
            if asterisk_fallback not in matched_ingredients:
                matched_ingredients.append(asterisk_fallback)

    # --- Strict Rule: Ignore entry if no ingredients could be fetched/parsed ---
    if not matched_ingredients:
        print(f"  ❌ Skipping recipe '{recipe['title']}': No valid ingredients gathered.")
        continue

    # Build the normalized entry
    normalized_entry = {
        "id": recipe["id"],
        "title": recipe["title"],
        "source": recipe["source"],
        "videoUrl": recipe["videoUrl"],
        "ingredients": matched_ingredients,
        "prepTime": recipe["prepTime"],
        "servings": recipe["servings"],
    }
    final_data["recipes"].append(normalized_entry)

# 4. Save to final destination file
print("\n" + "="*60 + "\n💾 Saving processed files...")
with open(output_recipe_path, "w", encoding="utf-8") as json_file:
    json.dump(final_data, json_file, ensure_ascii=False, indent=4)

print(f"🎉 Mapping Complete! Check out your updated entries in '{output_recipe_path}'.")

📋 Loading master ingredients database...
🔄 Starting verbose mapping for 10 recipes...

[#1/10] Mapping: Daging Dendeng
  ✅ [MATCH] '1 kg daging (direbus 45 minit)' ➡️ 'daging'
  ✅ [MATCH] '4 btg serai' ➡️ 'serai'
  ✅ [MATCH] 'Halia sebesar ibu jari' ➡️ 'halia'
  ✅ [MATCH] '4 biji bawang merah' ➡️ 'bawang-merah'
  ✅ [MATCH] '6 ulas bawang putih' ➡️ 'bawang-putih'
  ✅ [MATCH] 'Segenggam cili kering yang direbus' ➡️ 'cili-kering'
  ✅ [MATCH] '3 helai daun limau' ➡️ 'daun-limau'
  ✅ [MATCH] '1 cawan kicap manis' ➡️ 'kicap-manis'
  ✅ [MATCH] 'Gula' ➡️ 'gula'
  ✅ [MATCH] 'Garam' ➡️ 'garam'
  ✅ [MATCH] 'Ajinomoto' ➡️ 'ajinomoto'
  ✅ [MATCH] '6 cili padi merah' ➡️ 'cili-padi'
  ✅ [MATCH] 'Sedikit asam jawa' ➡️ 'asam-jawa'

[#2/10] Mapping: Rendang 2 Kilo Daging
  ✅ [MATCH] '4 biji bawang besar' ➡️ 'bawang-besar'
  ✅ [MATCH] '2 ketul halia' ➡️ 'halia'
  ✅ [MATCH] '8 batang serai' ➡️ 'serai'
  ✅ [MATCH] '2 ketul lengkuas' ➡️ 'lengkuas'
  ✅ [MATCH] '8 ulas bawang putih' ➡️ 'bawang-putih'
  ✅ [MAT